# Week 11 Benchmark Analysis

Use only the supplied synthetic data. Preserve the raw CSV unchanged and perform every transformation in code.

## Analysis question

**Which clinics and referral services have the lowest on-time follow-up completion rates, and where should the program focus its next quality-improvement review?**

For this assignment:

- The analysis snapshot is `2026-10-31`.
- A referral is **completed** when `completion_date` is present.
- A referral is **on time** when it has a completion date on or before its due date.
- The required on-time rate uses **all referrals in the group** as the denominator. Pending referrals therefore remain in the denominator.

Work through the notebook in order. Each section separates observation, transformation, verification, and interpretation so that your decisions remain traceable. Complete every TODO, record cleaning decisions in the cleaning log, and restart the runtime and run all cells before submitting.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW_PATH = Path("Week11_Messy_Followup_Data.csv")
CLEAN_PATH = Path("Week11_Cleaned_Followup_Data.csv")
SNAPSHOT = pd.Timestamp("2026-10-31")


## 1 Load the raw data and create a working copy

The raw file is evidence of what was received. Do not overwrite it or clean it manually. Load it as `df_raw`, then create a separate copy named `df` for all cleaning and analysis.

The analytic grain is **one row per referral record**. A patient may legitimately have more than one referral, so `patient_id` is not the duplicate key. `record_id` identifies the synthetic source referral row.


In [ ]:
# TODO 1: Read RAW_PATH into df_raw.
# Load record_id and patient_id as pandas string columns so their identifiers
# are not interpreted as numbers.
df_raw = None

# TODO 2: Make an independent working copy named df.
df = None

# These lines should report 42 rows and 9 columns after the TODOs are complete.
print("Raw shape:", df_raw.shape)
print("Working shape:", df.shape)
print("Separate DataFrame objects:", df is not df_raw)


### 1.1 Inspect structure and data types

Start with the dataset's size, column names, first records, and data types. Do not assume that a date-looking or number-looking column was imported with the correct type.

Questions to consider:

- Are all fields from the data dictionary present?
- Which date fields were imported as text?
- Which numeric field may contain a value that prevents numeric import?


In [ ]:
# Run this cell before cleaning.
print("Rows and columns:", df_raw.shape)
print("Column names:", df_raw.columns.tolist())
display(df_raw.head())

print("\nImported data types:")
display(df_raw.dtypes.to_frame("dtype"))


### 1.2 Quantify missingness and possible duplicates

Use more than one duplicate check:

- `df_raw.duplicated()` identifies rows that are exact copies across every column.
- Repeated `record_id` values help verify which referral rows were duplicated.
- Do **not** remove records merely because `patient_id` repeats; a patient can have multiple valid referrals.


In [ ]:
missing_before = df_raw.isna().sum().to_frame("missing_count")
missing_before["missing_percent"] = (
    missing_before["missing_count"] / len(df_raw) * 100
)
display(missing_before)

exact_duplicate_count = int(df_raw.duplicated().sum())
print("Exact duplicate rows:", exact_duplicate_count)

duplicate_record_ids = df_raw.loc[
    df_raw["record_id"].duplicated(keep=False), "record_id"
].value_counts()
print("\nRepeated record_id values:")
display(duplicate_record_ids)


### 1.3 Examine category and numeric values

Category variants may differ only by capitalization or spacing. A frequency table makes these differences easier to see than scanning the full dataset. Include missing values so they are not silently excluded.

Also inspect `contact_attempts`. A text value in that field is not automatically equivalent to a number, even when a reader can infer what the text probably means.


In [ ]:
for column in ["clinic", "referral_service", "social_needs_flag", "contact_attempts"]:
    print(f"\n{column}")
    display(df_raw[column].value_counts(dropna=False).to_frame("rows"))


### 1.4 Examine the original date patterns

The three date fields contain more than one text format. Inspect representative values before parsing them. Later, `errors="coerce"` will convert values that cannot be parsed to missing timestamps (`NaT`), which lets you count and review failures rather than silently treating invalid text as a date.


In [ ]:
date_columns = ["referral_date", "due_date", "completion_date"]

for column in date_columns:
    examples = sorted(df_raw[column].dropna().astype(str).unique())
    print(f"\n{column}: {len(examples)} distinct nonmissing values")
    print("First five examples:", examples[:5])
    print("Last five examples:", examples[-5:])


### Pre-cleaning observations

Complete this table before changing the data. Your cleaning log should contain more detail, including the number of rows affected and how you verified each decision.

| Issue | Evidence observed in the raw data | Planned handling |
|---|---|---|
| Exact duplicates | [Your observation] | Remove only exact duplicate rows. |
| Clinic labels | [List or describe the variants] | Map variants to the three approved clinic names. |
| Service labels | [List or describe the variants and missingness] | Standardize case; use `Unknown` for a missing service. |
| Date formats | [Describe the patterns] | Parse all three date fields; unparseable values become `NaT`. |
| Contact attempts | [Identify the unexpected value] | Convert to numeric; unparseable values become missing. |

**Why is `patient_id` not an appropriate duplicate key for this dataset?**

[Your response]


## 2 Clean and document the data

Clean one issue at a time. After each transformation, inspect or count the result before moving forward. Keep the original columns `clinic` and `referral_service`; create `clinic_clean` and `service_clean` so the standardized values can be compared with the source values.


### 2.1 Remove exact duplicate rows

Use `drop_duplicates()` on the complete row. Calculate the number removed by comparing the row count immediately before and after the operation. The raw `df_raw` object should remain unchanged.


In [ ]:
rows_before_deduplication = len(df)

# TODO: Remove exact duplicate rows from df and use .copy().

rows_after_deduplication = len(df)
rows_removed = rows_before_deduplication - rows_after_deduplication

print("Rows before:", rows_before_deduplication)
print("Rows after:", rows_after_deduplication)
print("Exact duplicate rows removed:", rows_removed)
print("Remaining repeated record_id values:", int(df["record_id"].duplicated().sum()))


### 2.2 Standardize clinic and service labels

The approved clinic labels are:

- `Northside Clinic`
- `Riverbend Clinic`
- `Mountain View Clinic`

The approved named service labels are:

- `Medication review`
- `Social work`
- `Nutrition`
- `Care coordination`

Use mapping dictionaries rather than a long series of row-by-row conditionals. Strip extra spaces and use case-insensitive matching where appropriate. A missing service is a data-quality issue, but the referral must remain in the overall total; label that service `Unknown`.


In [ ]:
# TODO: Add every raw clinic variant observed in Section 1 as a key and map it
# to one of the three approved clinic names.
clinic_map = {
    # "raw clinic label": "Approved Clinic Label",
}

approved_services = [
    "Medication review",
    "Social work",
    "Nutrition",
    "Care coordination",
]
service_map = {label.lower(): label for label in approved_services}

# TODO: Create clinic_clean by stripping clinic text and applying clinic_map.

# TODO: Create service_clean by filling missing source values with an empty
# string, stripping spaces, converting to lowercase, applying service_map,
# and finally replacing any remaining missing result with "Unknown".


#### Category-mapping checkpoint

Run this after creating the cleaned category columns. An empty unmatched-clinic table confirms that every observed clinic variant was mapped. The clinic cross-tab also documents exactly how raw labels changed.


In [ ]:
unmatched_clinics = df.loc[df["clinic_clean"].isna(), "clinic"].value_counts(dropna=False)
print("Unmatched clinic labels (should be empty):")
display(unmatched_clinics)

print("Raw-to-clean clinic mapping:")
display(pd.crosstab(df["clinic"], df["clinic_clean"], dropna=False))

print("Raw-to-clean service mapping:")
display(pd.crosstab(df["referral_service"].fillna("<missing>"), df["service_clean"], dropna=False))


### 2.3 Parse the date fields

Convert `referral_date`, `due_date`, and `completion_date` to pandas datetime values. Use `format="mixed"` because the supplied data contain both `MM/DD/YYYY` and `YYYY-MM-DD`. Use `errors="coerce"` so invalid text becomes `NaT` and can be reviewed.

Compare missing counts before and after parsing. For `completion_date`, preexisting blanks are meaningful: they represent referrals not completed at the snapshot. A rise in missingness after parsing would indicate an additional unparseable value.


In [ ]:
date_missing_before = df[date_columns].isna().sum()

for column in date_columns:
    # TODO: Parse df[column] with pd.to_datetime, format="mixed",
    # and errors="coerce".
    pass

date_missing_after = df[date_columns].isna().sum()
date_parse_check = pd.DataFrame({
    "missing_before": date_missing_before,
    "missing_after": date_missing_after,
    "new_missing_after_parse": date_missing_after - date_missing_before,
})
display(date_parse_check)
print("Parsed date data types:")
display(df[date_columns].dtypes.to_frame("dtype"))


### 2.4 Convert contact attempts to numeric

Preserve a copy of the original values for verification. Then use `pd.to_numeric(..., errors="coerce")`. Do not replace an unexpected text value with an inferred number; converting it to missing makes the uncertainty visible.


In [ ]:
contact_attempts_raw = df["contact_attempts"].copy()
missing_contact_before = int(contact_attempts_raw.isna().sum())

# TODO: Convert df["contact_attempts"] to numeric with errors="coerce".

missing_contact_after = int(df["contact_attempts"].isna().sum())
new_invalid_contact_values = missing_contact_after - missing_contact_before

print("Missing before numeric conversion:", missing_contact_before)
print("Missing after numeric conversion:", missing_contact_after)
print("New missing values caused by conversion:", new_invalid_contact_values)
print("Original values that did not parse:")
display(contact_attempts_raw[df["contact_attempts"].isna()].value_counts(dropna=False))


### 2.5 Create completion and on-time indicators

Create two Boolean fields:

- `completed` is `True` when `completion_date` is not missing.
- `on_time` is `True` only when the referral is completed **and** `completion_date` is on or before `due_date`.

The required rate later divides the number of `True` values in `on_time` by **all referrals in the group**, not only completed referrals. This is why it is important to keep `completed` and `on_time` as separate fields.


In [ ]:
# TODO: Create the completed Boolean field using .notna().

# TODO: Create the on_time Boolean field using both the completed indicator
# and the comparison completion_date <= due_date.

print("Completed value counts:")
display(df["completed"].value_counts(dropna=False))
print("On-time value counts:")
display(df["on_time"].value_counts(dropna=False))
print("Completed/on-time cross-check:")
display(pd.crosstab(df["completed"], df["on_time"], margins=True))


### 2.6 Required cleaning checks

These checks verify the minimum expected structure. Do not edit the expected values. If a check fails, return to the relevant cleaning step rather than deleting the assertion.


In [ ]:
# Required checks: do not change the expected values.
assert df_raw.shape[0] == 42
assert int(df_raw.duplicated().sum()) == 2
assert df.shape[0] == 40
assert df["record_id"].nunique() == 40

assert set(df["clinic_clean"].dropna().unique()) == {
    "Northside Clinic", "Riverbend Clinic", "Mountain View Clinic"
}
assert set(df["service_clean"].unique()) == {
    "Medication review", "Social work", "Nutrition",
    "Care coordination", "Unknown"
}
assert all(pd.api.types.is_datetime64_any_dtype(df[column]) for column in date_columns)
assert pd.api.types.is_numeric_dtype(df["contact_attempts"])
assert pd.api.types.is_bool_dtype(df["completed"])
assert pd.api.types.is_bool_dtype(df["on_time"])
assert not (df["on_time"] & ~df["completed"]).any()

print("Core cleaning checks passed.")


### Cleaning-log checkpoint

Before continuing, update `Week11_Cleaning_Log.md`. For each issue, record:

1. The evidence you observed.
2. The decision you made.
3. The number of rows affected.
4. The code or check that verified the result.

Include exact duplicates, clinic labels, service labels and missing service, date parsing, and contact-attempt conversion.


## 3 Calculate descriptive summaries

Build the analysis from individual quantities before grouping the data.

For the required measure:

\[
\text{on-time rate} = \frac{\text{referrals completed on or before the due date}}{\text{all referrals in the analyzed group}}
\]

Do not divide by completed referrals for the required rate. If you calculate that alternative rate for exploration, give it a different name and explain its denominator.


In [ ]:
# TODO: Calculate each component separately.
overall_referrals = None
overall_completed = None
overall_on_time = None
overall_on_time_rate = None

overall = {
    "referrals": overall_referrals,
    "completed": overall_completed,
    "on_time": overall_on_time,
    "on_time_rate": overall_on_time_rate,
}

display(pd.Series(overall, name="overall"))


### 3.1 Create a reusable grouped-summary function

The same calculations are required for clinics, services, and the exploratory social-needs comparison. A function reduces repeated code and keeps the denominator consistent.

Inside the function:

1. Group by the requested column with `dropna=False`.
2. Count referrals using `record_id`.
3. Sum the Boolean `completed` and `on_time` fields; in pandas, `True` counts as 1.
4. Divide each group's on-time count by that group's referral count.
5. Sort from lowest to highest observed on-time rate.


In [ ]:
def summarize(group_column):
    summary = (
        df.groupby(group_column, dropna=False)
          .agg(
              referrals=("record_id", "count"),
              completed=("completed", "sum"),
              on_time=("on_time", "sum"),
          )
    )

    # TODO: Add on_time_rate = on_time / referrals.

    # TODO: Sort summary from lowest to highest on_time_rate.

    return summary


clinic_summary = summarize("clinic_clean")
service_summary = summarize("service_clean")

print("Clinic summary:")
display(clinic_summary)
print("Service summary:")
display(service_summary)


### 3.2 Reconcile grouped totals to the overall result

Reconciliation checks whether records disappeared during grouping. Each summary should account for all referrals, completed referrals, and on-time referrals. This is especially important for the service summary because one source service is missing and must remain represented as `Unknown`.


In [ ]:
# Required reconciliation checks: do not change these comparisons.
for name, summary in {
    "clinic": clinic_summary,
    "service": service_summary,
}.items():
    assert int(summary["referrals"].sum()) == overall_referrals, f"{name} referral total does not reconcile"
    assert int(summary["completed"].sum()) == overall_completed, f"{name} completed total does not reconcile"
    assert int(summary["on_time"].sum()) == overall_on_time, f"{name} on-time total does not reconcile"

print("Clinic and service totals reconcile to the overall result.")


### 3.3 Explore the documented social-needs flag

Create a descriptive summary by `social_needs_flag` to support the required equity limitation. Treat this variable carefully: it indicates that a need was **documented**, not that it measures every person's circumstances or explains an outcome.

When interpreting the table, inspect group sizes and consider missing or unmeasured factors. Do not describe documented social needs as causing lower or higher performance.


In [ ]:
# TODO: Use summarize() to create social_needs_summary.
social_needs_summary = None

display(social_needs_summary)

# TODO: Reconcile the referral total in social_needs_summary to overall_referrals.


## 4 Create the visualizations

Create two bar charts that answer different parts of the analysis question:

1. On-time rate by clinic.
2. On-time rate by referral service.

Both charts need a zero baseline, a 0-to-1 y-axis, percent formatting, descriptive titles, labeled axes, readable category labels, and sufficient contrast. The service chart should show `Unknown` because excluding it would hide a data-quality issue; discuss why the one-record group should not be interpreted as meaningful performance evidence.


In [ ]:
# Clinic chart
clinic_rates = clinic_summary["on_time_rate"].sort_values()

# TODO: Plot clinic_rates as a bar chart.
# Suggested structure:
# ax = clinic_rates.plot(kind="bar", color="#4C78A8", ylim=(0, 1))
# Then add a factual title, x-axis label, y-axis label, percent formatting,
# readable x labels, and tight layout.

# TODO: Save as Week11_Clinic_Rates.png with dpi=150, then display the chart.


In [ ]:
# Service chart
service_rates = service_summary["on_time_rate"].sort_values()

# TODO: Plot service_rates as a bar chart with a different accessible color.
# Use the same 0-to-1 scale and percent formatting as the clinic chart so the
# rates are easy to compare.

# TODO: Save as Week11_Service_Rates.png with dpi=150, then display the chart.


## 5 Extract evidence before interpreting

Use the summary tables to identify the lowest observed clinic rate. For the service comparison, report `Unknown` as a data-quality limitation but do not treat its single record as a meaningful service-performance finding. Compare the four named services when answering which service has the lowest observed rate.

Display the numerator, denominator, and rate—not only the rate—so the evidence remains interpretable.


In [ ]:
# TODO: Sort clinic_summary and display the row with the lowest on_time_rate.
lowest_clinic_evidence = None

# Exclude Unknown only for the named-service comparison. Keep it in the full
# service summary and overall denominator.
named_service_summary = service_summary.loc[
    service_summary.index != "Unknown"
]

# TODO: Sort named_service_summary and display the row(s) with the lowest
# on_time_rate. Use more than one row if named services are tied.
lowest_service_evidence = None

print("Lowest clinic evidence:")
display(lowest_clinic_evidence)
print("Lowest named-service evidence:")
display(lowest_service_evidence)


### Interpretation and recommendation

Write concise answers in your own words. Use counts and rates from the summaries.

1. Which clinic has the lowest observed on-time rate? Give its on-time numerator, referral denominator, and percentage.
2. Which named service has the lowest observed on-time rate? If the rates are tied or very similar, say so rather than overstating a difference.
3. How should the `Unknown` service group be described?
4. What does this descriptive analysis **not** establish?
5. Explain at least two limitations. One must address the documented social-needs comparison, including data quality, group size, documentation, or unmeasured factors.
6. Recommend one proportionate next review step supported by the evidence. Focus on reviewing workflow or documentation before proposing an intervention.

**Your interpretation:**

[Write your response here.]


## 6 Document AI use and verification

AI may help with a specified coding or visualization task, but you remain responsible for the result. Record enough detail for another person to understand what the tool contributed and how you tested it.

- **Tool:**
- **Task and exact prompt:**
- **Relevant response or suggestion:**
- **Verification test:**
- **Expected result:**
- **Actual result:**
- **Revision and your judgment:**

Do not enter real patient or workplace data into an AI tool.


## 7 Export the cleaned file

Export a copy rather than changing the analysis DataFrame solely for presentation. Format parsed dates as `YYYY-MM-DD` in the exported file. Missing dates will remain blank in the CSV.


In [ ]:
export_df = df.copy()

for column in date_columns:
    # TODO: Convert the parsed datetime column to YYYY-MM-DD strings with
    # .dt.strftime("%Y-%m-%d").
    pass

# TODO: Export export_df to CLEAN_PATH without the pandas index.

print("Cleaned file path:", CLEAN_PATH)
print("Cleaned file created:", CLEAN_PATH.exists())
print("Rows prepared for export:", len(export_df))

# Required export checks. Run them after completing the TODOs above.
assert CLEAN_PATH.exists(), "The cleaned CSV was not created."
export_check = pd.read_csv(CLEAN_PATH)
assert len(export_check) == 40, "The cleaned CSV should contain 40 rows."
print("Export checks passed.")


## Final reproducibility checklist

Before submitting:

- Confirm that `Week11_Messy_Followup_Data.csv` is unchanged.
- Complete the cleaning log with evidence, decisions, affected rows, and verification.
- Confirm that clinic, service, and social-needs summaries reconcile to the overall total.
- Confirm that both charts use percent scales beginning at zero.
- Confirm that the interpretation reports numerators and denominators and avoids causal claims.
- Complete the AI-use record, even if no AI tool was used.
- Restart the runtime and run every notebook cell from top to bottom.
- Confirm that `Week11_Cleaned_Followup_Data.csv` is recreated successfully.
